<a href="https://colab.research.google.com/github/AHichem/bridge-deterioration-nbi/blob/main/notebooks/02_sql_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 2 — Modèle de données SQL

Objectif : à partir de la table `inspections` nettoyée, construire un **panel pont × année** avec les événements de dégradation, plus les tables de dimensions.

- **Partie 0** (fournie) : setup + chargement/nettoyage → table `inspections`
- **Étapes 1 à 7** : c'est toi qui écris le SQL

## 0.1 — Setup

In [1]:
!pip install -q jupysql duckdb-engine

from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/nbi'

import duckdb
con = duckdb.connect(f'{DATA_DIR}/nbi.duckdb')

%load_ext sql
%sql con
%config SqlMagic.displaylimit = 20

Mounted at /content/drive


## 0.2 — Chargement + nettoyage → `inspections`

Deux temps :
1. **Lecture (Python/pandas)** → `inspections_raw` : les 19 colonnes utiles, en texte brut, une année à la fois. On passe par pandas car le lecteur CSV de DuckDB n'arrive pas à lire certains fichiers NBI (fins de ligne mélangées `\n` / `\r\n`, apostrophes dans les textes) ; pandas est plus tolérant.
2. **Nettoyage (SQL)** → `inspections` : typage, conversion lat/lon, notes `'N'` → NULL, `min_rating`, `age`.

| Colonne | Contenu |
|---|---|
| `bridge_id` | `state_code` + `-` + `structure_number` |
| `year` | année du fichier NBI (2016–2025 ; le fichier 2015 téléchargé n'est pas exploitable) |
| `state_code`, `county_code`, `lat`, `lon` | localisation (lat/lon en degrés décimaux, lon négative) |
| `owner_code`, `material_code`, `structure_type_code` | codes bruts (items 22, 43A, 43B) |
| `year_built`, `year_reconstructed` | entiers, NULL si 0/absent |
| `adt`, `pct_truck`, `length_m`, `deck_width_m` | trafic et géométrie |
| `deck`, `super`, `sub`, `culvert` | notes 0–9, `'N'` → NULL |
| `is_culvert` | `culvert IS NOT NULL` |
| `min_rating` | `culvert` si ponceau, sinon `LEAST(deck, super, sub)` |
| `age` | `year - year_built` |

Choix de nettoyage :
- on garde seulement `RECORD_TYPE_005A = '1'` (les lignes de type 2 décrivent la route *sous* le pont et dupliqueraient la structure) ;
- les doublons éventuels ne sont **pas** supprimés ici : tu les chercheras à l'étape 6.

⏱️ Lecture de ~2,5 Go depuis le Drive : compte plusieurs minutes.

In [2]:
# Lecture des fichiers bruts avec pandas, un fichier (= une année) à la fois.
# pandas tolère les fins de ligne mélangées (\n / \r\n) qui font planter le lecteur CSV de DuckDB.
import glob, os, re
import pandas as pd

COLS = [
    'STATE_CODE_001', 'STRUCTURE_NUMBER_008', 'RECORD_TYPE_005A', 'COUNTY_CODE_003',
    'LAT_016', 'LONG_017', 'OWNER_022', 'STRUCTURE_KIND_043A', 'STRUCTURE_TYPE_043B',
    'YEAR_BUILT_027', 'YEAR_RECONSTRUCTED_106', 'ADT_029', 'PERCENT_ADT_TRUCK_109',
    'STRUCTURE_LEN_MT_049', 'DECK_WIDTH_MT_052',
    'DECK_COND_058', 'SUPERSTRUCTURE_COND_059', 'SUBSTRUCTURE_COND_060', 'CULVERT_COND_062',
]

con.execute('DROP TABLE IF EXISTS inspections_raw')
for f in sorted(glob.glob(f'{DATA_DIR}/raw/nbi_20*.txt')):
    year = int(re.search(r'nbi_(20\d{2})', f).group(1))
    if year < 2016:          # nbi_2015.txt n'est pas un vrai fichier NBI
        continue
    df = pd.read_csv(
        f, sep=',', quotechar="'", encoding='latin-1',
        dtype=object, keep_default_na=False, na_values=[''],   # tout en texte, vide -> NULL
        usecols=lambda c: c.strip() in COLS,
        on_bad_lines='warn',                                # signale les lignes illisibles au lieu de planter
    )
    df.columns = df.columns.str.strip()
    df = df.reindex(columns=COLS)                           # colonne absente une année -> NULL
    df.insert(0, 'year', year)
    if con.sql("SELECT COUNT(*) FROM information_schema.tables WHERE table_name = 'inspections_raw'").fetchone()[0] == 0:
        con.execute('CREATE TABLE inspections_raw AS SELECT * FROM df')
    else:
        con.execute('INSERT INTO inspections_raw SELECT * FROM df')
    print(f'{year} : {len(df):>8,} lignes')
    del df

print(con.sql('SELECT COUNT(*) AS total FROM inspections_raw').fetchone()[0], 'lignes au total')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2016 :  614,387 lignes


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2017 :  615,002 lignes


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

2023 :  621,581 lignes


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2024 :  623,218 lignes


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2025 :  624,193 lignes
6190306 lignes au total


In [3]:
%%sql
-- Nettoyage + typage : inspections_raw (texte brut) -> inspections (table propre)
CREATE OR REPLACE TABLE inspections AS
WITH typed AS (
    SELECT
        lpad(trim(STATE_CODE_001), 2, '0') || '-' || trim(STRUCTURE_NUMBER_008) AS bridge_id,
        CAST(year AS INTEGER) AS year,
        lpad(trim(STATE_CODE_001), 2, '0')  AS state_code,
        lpad(trim(COUNTY_CODE_003), 3, '0') AS county_code,

        -- LAT_016 = DDMMSSss, LONG_017 = DDDMMSSss (ouest, donc négatif)
        lpad(trim(LAT_016), 8, '0')  AS lat_txt,
        lpad(trim(LONG_017), 9, '0') AS lon_txt,

        lpad(trim(OWNER_022), 2, '0')            AS owner_code,
        trim(STRUCTURE_KIND_043A)                AS material_code,
        lpad(trim(STRUCTURE_TYPE_043B), 2, '0')  AS structure_type_code,

        NULLIF(TRY_CAST(trim(YEAR_BUILT_027)         AS INTEGER), 0) AS year_built,
        NULLIF(TRY_CAST(trim(YEAR_RECONSTRUCTED_106) AS INTEGER), 0) AS year_reconstructed,

        TRY_CAST(trim(ADT_029)               AS INTEGER) AS adt,
        TRY_CAST(trim(PERCENT_ADT_TRUCK_109) AS INTEGER) AS pct_truck,
        TRY_CAST(trim(STRUCTURE_LEN_MT_049)  AS DOUBLE)  AS length_m,
        TRY_CAST(trim(DECK_WIDTH_MT_052)     AS DOUBLE)  AS deck_width_m,

        -- notes 0-9 ; 'N' (non applicable) ou vide -> NULL
        TRY_CAST(trim(DECK_COND_058)           AS TINYINT) AS deck,
        TRY_CAST(trim(SUPERSTRUCTURE_COND_059) AS TINYINT) AS super,
        TRY_CAST(trim(SUBSTRUCTURE_COND_060)   AS TINYINT) AS sub,
        TRY_CAST(trim(CULVERT_COND_062)        AS TINYINT) AS culvert
    FROM inspections_raw
    WHERE trim(RECORD_TYPE_005A) = '1'   -- 1 = la structure elle-même (2 = route qui passe dessous, doublon)
)
SELECT
    bridge_id, year, state_code, county_code,
    CASE WHEN TRY_CAST(lat_txt AS BIGINT) > 0 THEN
        CAST(substr(lat_txt, 1, 2) AS INTEGER)
      + CAST(substr(lat_txt, 3, 2) AS INTEGER) / 60.0
      + CAST(substr(lat_txt, 5, 4) AS INTEGER) / 100.0 / 3600.0
    END AS lat,
    CASE WHEN TRY_CAST(lon_txt AS BIGINT) > 0 THEN
      -( CAST(substr(lon_txt, 1, 3) AS INTEGER)
       + CAST(substr(lon_txt, 4, 2) AS INTEGER) / 60.0
       + CAST(substr(lon_txt, 6, 4) AS INTEGER) / 100.0 / 3600.0 )
    END AS lon,
    owner_code, material_code, structure_type_code,
    year_built, year_reconstructed,
    adt, pct_truck, length_m, deck_width_m,
    deck, super, sub, culvert,
    culvert IS NOT NULL AS is_culvert,
    CASE WHEN culvert IS NOT NULL THEN culvert
         ELSE LEAST(deck, super, sub)          -- LEAST ignore les NULL dans DuckDB
    END AS min_rating,
    year - year_built AS age
FROM typed;

Running query in 'DuckDBPyConnection'

Count
6190306


In [4]:
%%sql
-- Vérification rapide du chargement (pas l'étape 1 : juste s'assurer que la table existe)
SELECT COUNT(*) AS n_rows, MIN(year) AS first_year, MAX(year) AS last_year
FROM inspections;

Running query in 'DuckDBPyConnection'

n_rows,first_year,last_year
6190306,2016,2025


In [5]:
%%sql
SELECT * FROM inspections LIMIT 5;

Running query in 'DuckDBPyConnection'

bridge_id,year,state_code,county_code,lat,lon,owner_code,material_code,structure_type_code,year_built,year_reconstructed,adt,pct_truck,length_m,deck_width_m,deck,super,sub,culvert,is_culvert,min_rating,age
01-00000000000S702,2016,01,053,31.103083333333334,-87.57038888888889,62,5,05,1999,None,50,1,15.0,9.7,8,8,7,None,False,7,17
01-00000000000S703,2016,01,053,31.105611111111113,-87.56913888888889,62,5,01,2002,None,159,1,13.6,9.8,8,8,7,None,False,7,14
01-0000000000M0022,2016,01,113,32.29535555555555,-84.97722222222222,74,1,04,1942,None,375,35,43.0,9.0,5,5,6,None,False,5,74
01-000000883039900,2016,01,059,34.45166666666667,-87.97,67,5,05,1974,None,300,5,65.5,10.5,7,7,7,None,False,7,42
01-000001014002450,2016,01,079,34.805,-87.38166666666666,67,3,10,1937,1962,6200,10,1981.2,7.8,6,6,7,None,False,6,79


In [6]:
%%sql
-- SELECT * FROM (describe inspections) ;
summarize inspections

Running query in 'DuckDBPyConnection'

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
bridge_id,VARCHAR,01-00000000000S702,78-STX8205,618519,None,None,None,None,None,6190306,0.00
year,INTEGER,2016,2025,11,2020.5149522495333,2.872505728201136,2018,2021,2023,6190306,0.00
state_code,VARCHAR,01,78,57,None,None,None,None,None,6190306,0.00
county_code,VARCHAR,001,840,330,None,None,None,None,None,6190306,0.00
lat,DOUBLE,8.333333333333334e-06,100.67777500000001,963509,37.90582099365086,5.072872148217294,34.42585269310194,38.486497425756156,41.32347615086201,6190306,0.06
lon,DOUBLE,-1000.677775,-5.555555555555556e-06,994436,-91.80387881350576,14.116938991824409,-97.1433334092447,-90.23652605708484,-83.01638879500022,6190306,0.13
owner_code,VARCHAR,00,80,41,None,None,None,None,None,6190306,0.00
material_code,VARCHAR,0,A,11,None,None,None,None,None,6190306,0.00
structure_type_code,VARCHAR,00,22,23,None,None,None,None,None,6190306,0.00
year_built,INTEGER,2,2026,283,1975.3424833285947,26.352013807856064,1959,1976,1996,6190306,0.00


---
## Étape 1 — Prise en main de la table (15 min)

**Objectif :** répondre à 4 questions, une requête chacune.
1. Combien de lignes (= ponts inspectés) par année ?
2. Combien de ponts **distincts** au total sur toute la période ?
3. Distribution de `min_rating` par année (combien de ponts à 0, 1, …, 9 chaque année).
4. Part de ponceaux (`is_culvert`) — globale ou par année.

**Concepts :** `GROUP BY`, `COUNT(DISTINCT …)`, agrégats conditionnels `COUNT(*) FILTER (WHERE …)`.

**Indices (sans solution) :**
- Q1 et Q2 ne donnent pas le même ordre de grandeur que la somme des lignes : pourquoi ?
- Q3 : deux façons possibles — format « long » (`GROUP BY` sur deux colonnes) ou format « large » (une colonne par note avec `FILTER`). Essaie au moins le format large pour pratiquer `FILTER`.
- Q4 : un booléen se compte avec `FILTER (WHERE is_culvert)`, et une part = compte filtré / compte total. Attention à la division entière.
- Pense aux `NULL` dans `min_rating` : où vont-ils dans tes résultats ?

In [12]:
%%sql
-- Number of bridges inspected by year
SELECT COUNT (DISTINCT bridge_id) AS nb_bridges_pery, year FROM inspections
GROUP BY year,
ORDER BY year;

Running query in 'DuckDBPyConnection'

nb_bridges_pery,year
614383,2016
614998,2017
616092,2018
617078,2019
618455,2020
619620,2021
620666,2022
621579,2023
623216,2024
624191,2025


In [13]:
%%sql
-- Nb of bridges
SELECT COUNT (DISTINCT bridge_id) AS distinct_bridges FROM inspections;


Running query in 'DuckDBPyConnection'

distinct_bridges
662768


In [32]:
%%sql
-- Q3 — distribution de min_rating par année
SELECT year,
COUNT (min_rating) FILTER  (WHERE min_rating = 0) AS minrat_0,
COUNT (min_rating) FILTER  (WHERE min_rating = 1) AS minrat_1,
COUNT (min_rating) FILTER  (WHERE min_rating = 2) AS minrat_2,
COUNT (min_rating) FILTER  (WHERE min_rating = 3) AS minrat_3,
COUNT (min_rating) FILTER  (WHERE min_rating = 4) AS minrat_4,
COUNT (min_rating) FILTER  (WHERE min_rating = 5) AS minrat_5,
COUNT (min_rating) FILTER  (WHERE min_rating = 6) AS minrat_6,
COUNT (min_rating) FILTER  (WHERE min_rating = 7) AS minrat_7,
COUNT (min_rating) FILTER  (WHERE min_rating = 8) AS minrat_8,
COUNT (min_rating) FILTER  (WHERE min_rating = 9) AS minrat_9,
FROM inspections
GROUP BY year,
ORDER BY year;



Running query in 'DuckDBPyConnection'

year,minrat_0,minrat_1,minrat_2,minrat_3,minrat_4,minrat_5,minrat_6,minrat_7,minrat_8,minrat_9
2016,1064,472,2052,9412,35553,102861,171441,209330,70256,11835
2017,1071,479,1901,8984,35181,105218,174047,210340,66523,11174
2018,980,444,1804,8802,35021,109250,176421,210220,62373,10730
2019,971,413,1736,8394,34647,111567,179765,209777,59384,10428
2020,967,440,1671,8124,33827,112796,182191,211548,56813,10079
2021,898,426,1560,7895,32805,113387,184516,214233,54412,9490
2022,1147,393,1310,7627,32490,114094,187294,214899,52542,8873
2023,769,396,1248,7430,32560,114488,189568,215621,50990,8511
2024,799,412,1250,7341,32276,114084,192190,216500,49643,8723
2025,764,431,1209,7118,32160,112275,197447,215168,48731,8890


In [19]:
%%sql
SELECT min_rating, bridge_id FROM inspections
WHERE year=2017;

Running query in 'DuckDBPyConnection'

min_rating,bridge_id
7,01-00000000000S702
7,01-00000000000S703
5,01-0000000000M0022
7,01-000000883039900
6,01-000001014002450
5,01-000001331700710
4,01-000002
4,01-000004
7,01-000004504800350
4,01-000005


In [ ]:
%%sql
-- Q4 — part de ponceaux
